# MAT benchmark, Tier 1 share on Colab GPU: tasks estrogen-alpha

Runtime → Change runtime type → **T4 GPU**. Upload your `kaggle.json` when asked (it is only used to download the private project bundle). Run all cells; at the end download `tier1_runs.csv` and copy it into the local project's `kaggle/output/colab_estrogen-alpha/` folder, then run `python scripts/fetch_kaggle_results.py --no-fetch --kernels colab/colab_estrogen-alpha`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, os; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'cpus', os.cpu_count())

In [ ]:
!pip install -q kaggle rdkit torch_geometric threadpoolctl 2>&1 | tail -1

In [ ]:
from google.colab import files
import os, json
up = files.upload()   # choose kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('credential installed')

In [ ]:
!rm -rf /content/bundle && mkdir -p /content/bundle && kaggle datasets download -d dipurao/molbench-bundle -p /content/bundle --unzip
!ls /content/bundle && ls /content/bundle/project | head

In [ ]:
import sys, subprocess, time, shutil
WORK = '/content/bundle/project'
os.chdir(WORK)
SRC = os.path.join(WORK, 'src')
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC
os.environ['MOLBENCH_DEVICE'] = 'cuda'
import molbench; print('molbench', molbench.__version__)
print('tuning present:', os.path.exists('results/tuning/best_configs.json'))

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_models.py', '-q', '-p', 'no:cacheprovider'], capture_output=True, text=True)
print(r.stdout[-600:]); assert r.returncode == 0

## Run the Tier-1 share (all 8 models, both splits, 5 seeds × 5 folds) — resume-safe; re-run the cell to continue

In [ ]:
os.makedirs('results/raw', exist_ok=True)
cmd = [sys.executable, '-u', 'scripts/03_run_tier1.py', '--tasks'] + ['estrogen-alpha'] + ['--workers', str(max(1, os.cpu_count())), '--threads', '1', '--out', 'results/raw/tier1_colab.csv']
print(' '.join(cmd))
p = subprocess.Popen(cmd, stdout=open('results/tier1_colab.log', 'w'), stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED='1'))
t0 = time.time()
while p.poll() is None:
    time.sleep(120)
    n = (sum(1 for _ in open('results/raw/tier1_colab.csv')) - 1) if os.path.exists('results/raw/tier1_colab.csv') else 0
    print(f'{(time.time()-t0)/60:5.1f} min  completed runs: {n}', flush=True)
print('exit code', p.returncode); print(open('results/tier1_colab.log').read()[-1500:])

In [ ]:
import pandas as pd
df = pd.read_csv('results/raw/tier1_colab.csv')
print(len(df), 'rows;', int((df['error'].fillna('') != '').sum()), 'errors')
print(df.groupby(['task', 'model']).size().unstack(fill_value=0))
out = '/content/tier1_runs.csv'
shutil.copy('results/raw/tier1_colab.csv', out)
from google.colab import files
files.download(out)